# Simple Gurobi Battery Model with price as a simple function of AIL
version 2
\
Trying to use a simple beta cdf in this version
NOTE: beta function still not working. Trying to look at the Logistic function.

In [1]:
# Let's start off by loading in the packages
import pandas as pd
import datetime
import os
import timeit
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB
import numpy as np
from scipy.optimize import curve_fit
from scipy.stats import beta

In [2]:
# Loading in the data first.
# Let's load in the data here, and for now, let's make it just for four hours.

file_name = os.fsdecode("marco_data.csv")
workbook = pd.read_csv(file_name)
workbook = workbook.filter(items=["date", "he", "size","flexible","price","ail","mkt_price"]) # Gets the columns that we need


workbook = workbook.query('date=="2024-04-10"') # Remove this to take out the filter for the day.

# Getting the columns we need.
# merit_data = workbook.filter(items=["date", "he", "size","flexible","price","mkt_price","ail","datetime"]) # Gets the columns that we need
merit_data = workbook
# I added "datetime" above so that it will be easier to filter out the merit data when looping through.

FileNotFoundError: [Errno 2] No such file or directory: 'marco_data.csv'

In [17]:
#merit_data

,date,he,size,flexible,price,ail,mkt_price
0,2024-04-10,1,0.0124,N,0.00,9169.9227,68.48
1,2024-04-10,1,0.0261,N,0.00,9169.9227,68.48
2,2024-04-10,1,0.0351,N,0.00,9169.9227,68.48
3,2024-04-10,1,0.1579,N,0.00,9169.9227,68.48
4,2024-04-10,1,0.9010,N,0.00,9169.9227,68.48
...,...,...,...,...,...,...,...
6131,2024-04-10,24,16.0000,Y,999.99,8518.8169,45.04
6132,2024-04-10,24,16.0000,Y,999.99,8518.8169,45.04
6133,2024-04-10,24,80.0000,Y,999.99,8518.8169,45.04
6134,2024-04-10,24,85.0000,Y,999.99,8518.8169,45.04


In [18]:
# Adding in battery data:

#### BATTERY DATA ####

# Creating the parameters of the battery here, and if we want to make changes to our battery this is where we can adjust it.
batt_dict = {
    'max_charge_rate':250,
    'max_discharge_rate':250,
    'capacity':1000,
    'charge_eff':0.05,
    'discharge_eff':0.05,
    'min_soc':0.05,
    'max_soc':0.95,
    'initial_soc':200
}

battDF = pd.DataFrame.from_dict(batt_dict, orient = 'index').transpose()
del batt_dict
battDF

,max_charge_rate,max_discharge_rate,capacity,charge_eff,discharge_eff,min_soc,max_soc,initial_soc
0,250.0,250.0,1000.0,0.05,0.05,0.05,0.95,200.0


In [ ]:
# # Let's create a simple price function that we'll use in the gurobi model
# def simple_price(x):
#     return (0.5*x)

# Let's try using a simple beta function with predetermined alpha and beta parameters.
def simple_beta(x):
    return beta.cdf(x,0.5,0.5)

# NOTE: This would mean that x values that are fed into beta need to be normalized first.

In [ ]:
# And now let's create an index that will help us below.
time_index = merit_data['he'].unique()

# And then let's get the ail for each hour, as well as the maximum value so that it will help us with normalizing the x values.
ail_list = merit_data.groupby(by = 'he',as_index = False)['ail'].min()['ail']
ail_max = merit_data.groupby(by = 'he',as_index = False)['size'].sum()['size']

ail_normalized = []
for i in range(len(ail_list)):
    ail_normalized.append(ail_list[i]/ail_max[i])


[0.8099978193473576,
 0.7848351251722537,
 0.7713558366294005,
 0.7644893140535517,
 0.7725205027891748,
 0.7891466332314814,
 0.8121733803870791,
 0.7954852500898115,
 0.7952653126278653,
 0.7869836152066849,
 0.7940155395090235,
 0.7921889522596954,
 0.797672047158962,
 0.8001850543960755,
 0.7990999694913604,
 0.8029775729079994,
 0.7987052910204191,
 0.7845411971419011,
 0.8031961192140804,
 0.874812034208846,
 0.885733975274024,
 0.8618223994623371,
 0.8459811859299052,
 0.8054235012804277]

In [27]:
# Now we start with creating the gurobi model.

bm = gp.Model("qp")

inf = GRB.INFINITY

Set parameter Username
Academic license - for non-commercial use only - expires 2025-09-12


In [28]:
## Let's create the variables
# It's slightly different now because we're using Gurobi

# Create an index for the number of periods that's an integer; easier to use.
tIndex = len(time_index)

# Now creating the variables of a simpler battery
vBattPower = [bm.addVar(lb = -inf, ub = inf, name = "") for _ in range(tIndex)]
# Battery State of Charge Variable, which acts as a constraint as well.
vSOC = [bm.addVar(lb = battDF.iloc[0]['capacity']*battDF.iloc[0]['min_soc'], ub = battDF.iloc[0]['capacity']*battDF.iloc[0]['max_soc'], name = "") for _ in range(tIndex)]

# Simple Price function that governs prices:
vPrice = [bm.addVar(lb = -inf, ub = inf, name = "") for _ in range(tIndex)]

# Variable for the normalized vBattPower:
vBatt_Norm = [bm.addVar(lb = -inf, ub = inf, name = "") for _ in range(tIndex)]

In [29]:
## CONSTRAINTS

for i in range(tIndex):

    #---------------------------------------------------------------------------------------------------------------------------
    #### Battery Constraints:

    # Power flow constraints
    bm.addConstr(vBattPower[i] >= -battDF.iloc[0]['max_discharge_rate'])
    bm.addConstr(vBattPower[i] <= battDF.iloc[0]['max_charge_rate'])

    # Law of Motion for the state of charge of the battery:
    # NOTE: Because we are doing it this way, if BattPower is positive then our battery is charging.
    if i == 0:
        bm.addConstr(vSOC[i] == battDF.iloc[0]['initial_soc'] + vBattPower[i])
    else:
        bm.addConstr(vSOC[i] == vSOC[i-1] + vBattPower[i])

    #### Price Constraint:
    bm.addConstr(vBatt_Norm[i] == vBattPower[i]/ail_max[i])
    bm.addConstr(vPrice[i] == simple_beta(vBatt_Norm[i]))

    #---------------------------------------------------------------------------------------------------------------------------


    

NotImplementedError: 

In [25]:
#---------------------------------------------------------------
# Objective Function Setup

# Add objective
# Changed the objective to sum up the battery energy times the price, instead of power in the grid.
obj = 0
obj += sum([vPrice[i]*vBattPower[i] for i in range(tIndex)])

# Setting the objective
bm.setObjective(obj, GRB.MINIMIZE)
#---------------------------------------------------------------

In [26]:
# Command that tells gurobi to optimize the model.

bm.optimize()

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 11.0 (22631.2))

CPU model: AMD Ryzen 5 8645HS w/ Radeon 760M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 96 rows, 96 columns and 167 nonzeros
Model fingerprint: 0x096ec542
Model has 24 quadratic objective terms
Model has 24 general constraints
Variable types: 96 continuous, 0 integer (0 binary)
Coefficient statistics:
  Matrix range     [8e-05, 1e+00]
  Objective range  [0e+00, 0e+00]
  QObjective range [2e+00, 2e+00]
  Bounds range     [5e+01, 1e+03]
  RHS range        [8e-01, 3e+02]
Presolve removed 49 rows and 25 columns
Presolve time: 0.00s
Presolved: 96 rows, 96 columns, 286 nonzeros
Presolved model has 24 bilinear constraint(s)

Solving non-convex MIQCP

Variable types: 96 continuous, 0 integer (0 binary)
Found heuristic solution: objective -115.1594481

Root relaxation: objective -1.335884e+02, 78 iterations,

In [27]:
# Let's collect the battery power values and the price values:
prices = []
batt_power = []
state_of_charge = []

for i in range(tIndex):
    prices.append(100*vPrice[i].X)
    batt_power.append(vBattPower[i].X)
    state_of_charge.append(vSOC[i].X)


In [28]:
# Let's put our results in a table:

battery_results = pd.DataFrame()

battery_results['hour_ending'] = merit_data['he'].unique()
battery_results['simple_prices'] = prices
battery_results['batt_power'] = batt_power
battery_results['state_of_charge'] = state_of_charge

battery_results

,hour_ending,simple_prices,batt_power,state_of_charge
0,1,69.090938,-63.204426,136.795574
1,2,68.821181,79.918035,216.713609
2,3,68.675835,156.157551,372.871160
3,4,68.601540,197.342076,570.213236
4,5,68.688442,151.180639,721.393876
5,6,68.867511,55.279704,776.673580
6,7,69.113710,-75.316502,701.357078
7,8,68.935527,19.701175,721.058253
8,9,68.933244,21.774191,742.832444
9,10,68.844428,73.483291,816.315735
